In [1]:
# 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Install required packages
# %pip install --upgrade -q google-genai google-adk==1.9.0 a2a-sdk==0.3.0 python-dotenv aiohttp uvicorn requests mermaid-python nest-asyncio

In [ ]:

import asyncio
import logging
import os
import sys
import threading
import time

from typing import Any

import httpx
import nest_asyncio
import uvicorn

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    TransportProtocol,
    
)
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH
from dotenv import load_dotenv
from google.adk.a2a.executor.a2a_agent_executor import (
    A2aAgentExecutor,
    A2aAgentExecutorConfig,
)
from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search

# 2. Building Your A2A System
Let's build our three-agent system step by step. We'll create:

1. Trending Topics Agent - Finds current trending topics
2. Trend Analyzer Agent - Analyzes trends with quantitative data
3. Host Agent - Orchestrates the other agents (sequentially)

### Agent 1: Trending Topics Agent
This agent searches the web for trending topics and returns a list of current trends.

In [ ]:
# Create the Trending Topics ADK Agent
trending_agent = Agent(
    model='gemini-2.5-pro',
    name='trending_topics_agent',
    instruction="""
    You are a social media trends analyst. Your job is to search the web for current trending topics,
    particularly from social platforms.

    When asked about trends:
    1. Search for "trending topics today" or similar queries
    2. Extract the top 3 trending topics
    3. Return them in a JSON format

    Focus on current, real-time trends from the last 24 hours.

    You MUST return your response in the following JSON format:
    {
        "trends": [
            {
                "topic": "Topic name",
                "description": "Brief description (1-2 sentences)",
                "reason": "Why it's trending"
            },
            {
                "topic": "Topic name",
                "description": "Brief description (1-2 sentences)",
                "reason": "Why it's trending"
            },
            {
                "topic": "Topic name",
                "description": "Brief description (1-2 sentences)",
                "reason": "Why it's trending"
            }
        ]
    }

    Only return the JSON object, no additional text.
    """,
    tools=[google_search],
)

print('Trending Topics Agent created successfully!')

Trending Topics Agent created successfully!


In [5]:
trending_agent_card = AgentCard(
    name='Trending Topics Agent',
    url='http://localhost:10020',
    description='Searches the web for current trending topics from social media',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='find_trends',
            name='Find Trending Topics',
            description='Searches for current trending topics on social media',
            tags=['trends', 'social media', 'twitter', 'current events'],
            examples=[
                "What's trending today?",
                'Show me current Twitter trends',
                'What are people talking about on social media?',
            ],
        )
    ],
)
    

In [6]:
remote_trending_agent = RemoteA2aAgent(
    name='find_trends',
    description='Searches for current trending topics on social media',
    agent_card=f'http://localhost:10020{AGENT_CARD_WELL_KNOWN_PATH}',
)

C:\Users\skyop\AppData\Local\Temp\ipykernel_3496\303652596.py:1: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_trending_agent = RemoteA2aAgent(


### Agent 2: Trend Analyzer Agent
This agent takes a specific trend and performs deep analysis with quantitative data.

In [7]:
# Create the Trend Analyzer ADK Agent
analyzer_agent = Agent(
    model='gemini-2.5-pro',
    name='trend_analyzer_agent',
    instruction="""
    You are a data analyst specializing in trend analysis. When given a trending topic,
    perform deep research to find quantitative data and insights.

    For each trend you analyze:
    1. Search for statistics, numbers, and metrics related to the trend
    2. Look for:
       - Engagement metrics (views, shares, mentions)
       - Growth rates and timeline
       - Geographic distribution
       - Related hashtags or keywords
    3. Provide concrete numbers and data points

    Keep it somehow concise

    Always prioritize quantitative information over qualitative descriptions.
    """,
    tools=[google_search],
)

print('Trend Analyzer Agent created successfully!')
     

Trend Analyzer Agent created successfully!


In [8]:
analyzer_agent_card = AgentCard(
    name='Trend Analyzer Agent',
    url='http://localhost:10021',
    description='Performs deep analysis of trends with quantitative data',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='analyze_trend',
            name='Analyze Trend',
            description='Provides quantitative analysis of a specific trend',
            tags=['analysis', 'data', 'metrics', 'statistics'],
            examples=[
                'Analyze the #ClimateChange trend',
                'Get metrics for the Taylor Swift trend',
                'Provide data analysis for AI adoption trend',
            ],
        )
    ],
)

In [9]:
remote_analyzer_agent = RemoteA2aAgent(
    name='analyze_trend',
    description='Provides quantitative analysis of a specific trend',
    agent_card=f'http://localhost:10021{AGENT_CARD_WELL_KNOWN_PATH}',
)

C:\Users\skyop\AppData\Local\Temp\ipykernel_3496\738464179.py:1: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_analyzer_agent = RemoteA2aAgent(


### Agent 3: Host Agent (Orchestrator)

In [10]:
# Create the Host ADK Agent
host_agent = SequentialAgent(
    name='trend_analysis_host',
    sub_agents=[remote_trending_agent, remote_analyzer_agent],
)

In [ ]:
host_agent_card = AgentCard(
    name='Trend Analysis Host',
    url='http://localhost:10022',
    description='Orchestrates, sequentially, trend discovery and analysis using specialized agents',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['application/json'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='comprehensive_trend_analysis',
            name='Comprehensive Trend Analysis',
            description='Finds trending topics and provides deep analysis of the most relevant one',
            tags=['trends', 'analysis', 'orchestration', 'insights'],
            examples=[
                'Analyze current trends',
                "What's trending and why is it important?",
                'Give me a comprehensive trend report',
            ],
        )
    ],
)

### 3. Running
Now let's put everything together. We'll create helper functions to start our agents and run the complete system.

### Starting the A2A Servers
Create function to run each agent as an A2A server:

In [12]:
def create_agent_a2a_server(agent, agent_card):
    """Create an A2A server for any ADK agent.

    Args:
        agent: The ADK agent instance
        agent_card: The ADK agent card

    Returns:
        A2AStarletteApplication instance
    """
    runner = Runner(
        app_name=agent.name,
        agent=agent,
        artifact_service=InMemoryArtifactService(),
        session_service=InMemorySessionService(),
        memory_service=InMemoryMemoryService(),
    )

    config = A2aAgentExecutorConfig()
    executor = A2aAgentExecutor(runner=runner, config=config)

    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )

    # Create A2A application
    return A2AStarletteApplication(
        agent_card=agent_card, http_handler=request_handler
    )

In [ ]:
# Apply nest_asyncio
nest_asyncio.apply()

# Store server tasks
server_tasks: list[asyncio.Task] = []


async def run_agent_server(agent, agent_card, port) -> None:
    """Run a single agent server."""
    app = create_agent_a2a_server(agent, agent_card)

    config = uvicorn.Config(
        app.build(),
        host='127.0.0.1',
        port=port,
        log_level='warning',
        loop='none',  # Important: let uvicorn use the current loop
    )

    server = uvicorn.Server(config)
    await server.serve()


async def start_all_servers() -> None:
    """Start all servers in the same event loop."""
    # Create tasks for all servers
    tasks = [
        asyncio.create_task(
            run_agent_server(trending_agent, trending_agent_card, 10020)
        ),
        asyncio.create_task(
            run_agent_server(analyzer_agent, analyzer_agent_card, 10021)
        ),
        asyncio.create_task(
            run_agent_server(host_agent, host_agent_card, 10022)
        ),
    ]

    # Give servers time to start
    await asyncio.sleep(2)

    print('✅ All agent servers started!')
    print('   - Trending Agent: http://127.0.0.1:10020')
    print('   - Analyzer Agent: http://127.0.0.1:10021')
    print('   - Host Agent: http://127.0.0.1:10022')

    # Keep servers running
    try:
        await asyncio.gather(*tasks)
    except KeyboardInterrupt:
        print('Shutting down servers...')


# Run in a background thread


def run_servers_in_background() -> None:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_all_servers())


# Start the thread
server_thread = threading.Thread(target=run_servers_in_background, daemon=True)
server_thread.start()

# Wait for servers to be ready
time.sleep(3)

C:\Users\skyop\AppData\Local\Temp\ipykernel_3496\1072601589.py:19: UserWarning: [EXPERIMENTAL] A2aAgentExecutorConfig: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  config = A2aAgentExecutorConfig()
C:\Users\skyop\AppData\Local\Temp\ipykernel_3496\1072601589.py:20: UserWarning: [EXPERIMENTAL] A2aAgentExecutor: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  executor = A2aAgentExecutor(runner=runner, config=config)


✅ All agent servers started!
   - Trending Agent: http://127.0.0.1:10020
   - Analyzer Agent: http://127.0.0.1:10021
   - Host Agent: http://127.0.0.1:10022


c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\google\adk\a2a\executor\a2a_agent_executor.py:202: UserWarning: [EXPERIMENTAL] convert_a2a_request_to_agent_run_request: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  run_request = self._config.request_converter(
c:\Users\skyop\jaeho_template\dotenv\Lib\site-packages\google\adk\a2a\converters\request_converter.py:118: UserWarning: [EXPERIMENTAL] convert_a2a_part_to_genai_part: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subjected to breaking changes. A2A protocol and SDK arethemselves not experimental. Once it's stable enough the experimental mode will be remove

### 4. Testing the System

In [14]:
class A2ASimpleClient:
    """A2A Simple to call A2A servers."""

    def __init__(self, default_timeout: float = 240.0):
        self._agent_info_cache: dict[
            str, dict[str, Any] | None
        ] = {}  # Cache for agent metadata
        self.default_timeout = default_timeout

    async def create_task(self, agent_url: str, message: str) -> str:
        """Send a message following the official A2A SDK pattern."""
        # Configure httpx client with timeout
        timeout_config = httpx.Timeout(
            timeout=self.default_timeout,
            connect=10.0,
            read=self.default_timeout,
            write=10.0,
            pool=5.0,
        )

        async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
            # Check if we have cached agent card data
            if (
                agent_url in self._agent_info_cache
                and self._agent_info_cache[agent_url] is not None
            ):
                agent_card_data = self._agent_info_cache[agent_url]
            else:
                # Fetch the agent card
                agent_card_response = await httpx_client.get(
                    f'{agent_url}{AGENT_CARD_WELL_KNOWN_PATH}'
                )
                agent_card_data = self._agent_info_cache[agent_url] = (
                    agent_card_response.json()
                )

            # Create AgentCard from data
            agent_card = AgentCard(**agent_card_data)

            # Create A2A client with the agent card
            config = ClientConfig(
                httpx_client=httpx_client,
                supported_transports=[
                    TransportProtocol.jsonrpc,
                    TransportProtocol.http_json,
                ],
                use_client_preference=True,
            )

            factory = ClientFactory(config)
            client = factory.create(agent_card)

            # Create the message object
            message_obj = create_text_message_object(content=message)

            # Send the message and collect responses
            responses = []
            async for response in client.send_message(message_obj):
                responses.append(response)

            # The response is a tuple - get the first element (Task object)
            if (
                responses
                and isinstance(responses[0], tuple)
                and len(responses[0]) > 0
            ):
                task = responses[0][0]  # First element of the tuple

                # Extract text: task.artifacts[0].parts[0].root.text
                try:
                    return task.artifacts[0].parts[0].root.text
                except (AttributeError, IndexError):
                    return str(task)

            return 'No response received'

In [15]:
a2a_client = A2ASimpleClient()

In [16]:
async def test_trending_topics() -> None:
    """Test trending topics agent."""
    trending_topics = await a2a_client.create_task(
        'http://localhost:10020', "What's trending today?"
    )
    print(trending_topics)


# Run the async function
asyncio.run(test_trending_topics())

```json
{
    "trends": [
        {
            "topic": "#WWERaw",
            "description": "Topics and hashtags related to WWE's Monday Night Raw television broadcast are trending.",
            "reason": "Fans are reacting to the latest episode, including match outcomes, wrestler appearances, and storyline developments."
        },
        {
            "topic": "Amelia Earhart Files",
            "description": "Renewed discussion surrounding the files related to Amelia Earhart's disappearance.",
            "reason": "It's trending due to news reports that files supposedly declassified by the Trump administration were already public, sparking conversation online."
        },
        {
            "topic": "#GMMTV2026",
            "description": "The hashtag #GMMTV2026 is trending globally on X (formerly Twitter).",
            "reason": "Thai production company GMMTV held its \"GMMTV2026: UP & ABOVE PART 1\" event to announce its lineup of series and projects for the upcoming y

In [18]:
async def test_analysis() -> None:
    """Test analysis agent."""
    analysis = await a2a_client.create_task(
        'http://localhost:10021', 'Analyze the trend AI in Social Media'
    )
    print(analysis)


# Run the async function
asyncio.run(test_analysis())

### AI's Pervasive Influence on Social Media Quantified

Artificial intelligence is rapidly reshaping the social media landscape, with its market size in the sector projected to soar from approximately $2.4 billion in 2024 to an estimated $8.1 billion by 2030, reflecting a compound annual growth rate (CAGR) of 19.3%. Some projections are even more bullish, forecasting a market value of $12 billion by 2031. This growth is underpinned by the widespread integration of AI in content creation, user engagement, and marketing strategies.

**Content Generation and Engagement:**

A staggering 71% of images on social media are now AI-generated, signaling a monumental shift in content creation. This trend extends to text-based platforms, with estimates suggesting that 54% of long-form posts on LinkedIn are influenced by AI. On Reddit, approximately 13% of posts in 2024 were likely AI-generated, a 146% increase since 2021.

This surge in AI-driven content is directly impacting user engagement. Bus

In [19]:
async def test_host_analysis() -> None:
    """Test host analysis agent."""
    host_analysis = await a2a_client.create_task(
        'http://localhost:10022',
        'Find the most relevant trends in the web today, choose randomly one of the top '
        'trends, and give me a complete analysis of it with quantitative data',
    )
    print(host_analysis)


# Run the async function
asyncio.run(test_host_analysis())

An analysis of the trending hashtag #GMMTV2026 reveals significant online engagement and a strong digital footprint for the Thai entertainment company GMMTV, particularly leading up to its annual content lineup announcement. While specific metrics for the 2026 event are still emerging, data from previous years and the company's overall digital presence provide a quantitative look into the scale of this trend.

### Social Media Engagement & Reach:

The hashtag #GMMTV2026 has been a top worldwide trend on X (formerly Twitter), with one source indicating a massive volume of posts, pointing to a highly engaged global audience. The company's strategy heavily relies on social media metrics, with the CEO stating that engagement in the form of likes, shares, and comments is prioritized over traditional television ratings when casting and promoting artists.

**Instagram:**
*   **Followers:** GMMTV's official Instagram account has over 5.1 million followers. 
*   **Engagement:** The account sees